In [ ]:
# pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# IMPORTS
# FIRST, pip install -r requirements.txt

import json
import joblib
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import ipywidgets as widgets
from IPython.display import display, clear_output

W_CONTENT:   float = 0.80
W_COLLAB:    float = 0.20
FINAL_TOP_K: int   = 150

In [5]:
# PATHS
ROOT = Path(".")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"
RESULTS_DIR = ROOT / "results" / "playlists"

for d in (DATA_PROC, MODELS_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

USER_FILES = {
    "andy": DATA_RAW / "enrichment-success-andy.json",
    "dishita": DATA_RAW / "enrichment-success-dish.json",
    "riya": DATA_RAW / "enrichment-success-riya.json",
    "priyanka": DATA_RAW / "enrichment-success-priyanka.json",
}

USERS = list(USER_FILES.keys())

# TUNEABLE PARAMETERS FOR ENTIRE PIPELINE
AUDIO_FEATURES = [
    "danceability", "energy", "valence", "tempo", "loudness",
    "acousticness", "instrumentalness", "speechiness", "liveness",
    "key", "mode", "time_signature",
]
MOOD_FEATURES = ["valence", "energy", "danceability", "tempo"]

# DATA LOADING & PREPROCESSING | TRACK TABLES
MIN_MEANINGFUL_MS = 30_000
TFIDF_MIN_DF = 2
TFIDF_MAX_FEATURES = 500

# USER PROFILES
MIN_SKIP_RATE_DISLIKE = 0.7
MAX_MS_DISLIKE = 15_000
N_MOOD_CLUSTERS = 4

# CANDIDATE GENERATION
TOP_N_CONTENT = 300
TOP_N_TAG = 200
TOP_N_NEIGHBOR = 150
FINAL_CANDIDATE_K = 1000
TOP_K_TAGS = 10
MIN_NEIGHBOR_MS = 30_000
MAX_NEIGHBOR_SKIP = 0.3
NEIGHBOR_PLAY_THRESHOLD = 1 
PRE_SCORE_CONTENT_W = 0.50
PRE_SCORE_TAG_W = 0.20
PRE_SCORE_GROUP_W = 0.15
PRE_SCORE_NEIGHBOR_W = 0.15

In [6]:
# DATA LOADING AND PREPROCESSING

"""
Returns the best available tag list for a track.
 - Tries all_tags first.
 - If empty or missing, falls back to sc_genres.
 - If still empty, falls back to lastfm_tags.
 - Returns a list of lowercase, stripped strings.
 - If none are available, returns an empty list.
"""
def resolve_tags_columns(df: pd.DataFrame) -> pd.Series:
    tags = df["all_tags"]

    mask = tags.apply(lambda t: not isinstance(t, list) or len(t) == 0)
    tags = tags.where(~mask, df["sc_genres"])

    mask = tags.apply(lambda t: not isinstance(t, list) or len(t) == 0)
    tags = tags.where(~mask, df["lastfm_tags"])

    tags = tags.apply(
        lambda t: [x.strip().lower() for x in t] if isinstance(t, list) else []
    )

    return tags

"""
Loads a single user's raw (enriched) JSON listening history and returns it as a cleaned df.
 - Filters out non-track plays (episodes, audiobooks)
 - Renames audio feature columns to simplify access/keep consistency
 - Converts timestamps to pandas datetime
 - Ensures numeric values for ms_played
 - Marks skipped tracks as boolean
 - Adds 'tags' column
 - Adds 'meaningful' column
"""
def load_user_events(path: Path, user: str) -> pd.DataFrame:
    with open(path, encoding="utf-8") as f:
        raw = json.load(f)

    df = pd.json_normalize(raw)
    df['user'] = user

    df.rename(columns={f"audio_features.{f}": f for f in AUDIO_FEATURES}, inplace=True)
    mask = df["spotify_track_uri"].notna() & df["episode_name"].isna()
    df = df.loc[mask]

    df["ts"] = pd.to_datetime(df["ts"], utc=True)
    df["ms_played"] = pd.to_numeric(df["ms_played"], errors="coerce").fillna(0)
    df["skipped"] = df["skipped"].fillna(False).astype(bool)
    df["tags"] = resolve_tags_columns(df)

    # Meaningful play: not skipped AND listened to >= 30 seconds
    df["meaningful"] = (~df["skipped"]) & (df["ms_played"] >= MIN_MEANINGFUL_MS)
    return df

"""
Loads and combines listening history for all users into a single df.
Each row represents one play event, tagged with the user it belongs to.
"""
def load_all_events(user_files: dict) -> pd.DataFrame:
    frames = []

    for user, path in user_files.items():
        df = load_user_events(path, user)
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

events = load_all_events(USER_FILES)

# Save raw events for reference
events.to_csv(DATA_PROC / "events.csv", index=False)

print(f"Total play events: {len(events):,}")
print(f"Unique tracks: {events['spotify_track_uri'].nunique():,}")
print(f"Users: {events['user'].unique()}")

Total play events: 29,887
Unique tracks: 7,068
Users: <StringArray>
['andy', 'dishita', 'riya', 'priyanka']
Length: 4, dtype: str


In [7]:
# TRACK TABLE (ITEM VECTORS)

# COLUMNS TO INCLUDE IN THE TRACK TABLE
cols = [
    "spotify_track_uri",
    "master_metadata_track_name",
    "master_metadata_album_artist_name",
    "master_metadata_album_album_name",
    "tags",
    *AUDIO_FEATURES,
]

# BUILD TRACK TABLE: ONE ROW PER UNQIUE TRACK URI
track_table = (
    events[cols]
    .drop_duplicates("spotify_track_uri")
    .rename(columns={
        "master_metadata_track_name": "track_name",
        "master_metadata_album_artist_name": "artist_name",
        "master_metadata_album_album_name": "album_name",
    })
    .reset_index(drop=True)
)

# ASSIGN A UNIQUE INTEGER ID TO EACH TRACK
track_table["track_id"] = np.arange(len(track_table))

# LOOKUP DICTIONARIES (FOR FASTER MAPPING)
uri_to_id = track_table.set_index("spotify_track_uri")["track_id"].to_dict()
id_to_uri = track_table.set_index("track_id")["spotify_track_uri"].to_dict()

# NORMALIZED AUDIO FEATURE MATRIX
audio_matrix = track_table[AUDIO_FEATURES].fillna(0).to_numpy(dtype=float)
audio_scaler = MinMaxScaler()
audio_matrix_norm = audio_scaler.fit_transform(audio_matrix)

# TAG TF-IDF MATRIX
tag_docs = track_table["tags"].str.join(" ").fillna("")
tfidf = TfidfVectorizer(min_df=TFIDF_MIN_DF, max_features=TFIDF_MAX_FEATURES)
tag_matrix_sparse = tfidf.fit_transform(tag_docs)
tag_matrix = tag_matrix_sparse.toarray() # dense ver.
tag_vocab = tfidf.get_feature_names_out()

# CONVERT ARTIST NAME TO ARTIST ID
track_table["artist_id"] = pd.Categorical(track_table["artist_name"]).codes

# SAVE TRACK TABLE & FITTED MODELS
track_table.to_csv(DATA_PROC / "track_features.csv", index=False)
joblib.dump(audio_scaler, MODELS_DIR / "audio_scaler.joblib")
joblib.dump(tfidf, MODELS_DIR / "tfidf.joblib")

print(f"Track table saved: {len(track_table):,} tracks, tag vocab = {len(tag_vocab)}")

Track table saved: 7,068 tracks, tag vocab = 330


In [8]:
# USER PROFILES

# MAP TRACK URIS TO IDS
# Each track URI gets a unique integer ID for indexing into matrices.
events["track_id"] = events["spotify_track_uri"].map(uri_to_id)

# PER-(USER, TRACK) PLAY-STATS
"""
Aggregates raw play events to compute per-user, per-track statistics:
- n_plays: total plays of the track
- avg_ms: average milliseconds listened
- skip_count: number of skipped plays
- meaningful_pct: fraction of plays considered meaningful (>=30s, not skipped)
"""
play_stats = (
    events.groupby(["user", "track_id"])
    .agg(
        n_plays=("ms_played", "count"),
        avg_ms=("ms_played", "mean"),
        skip_count=("skipped", "sum"),
        meaningful_pct=("meaningful", "mean"),
    )
    .reset_index()
)

# Fraction of plays skipped for each user-track
play_stats["skip_rate"] = play_stats["skip_count"] / play_stats["n_plays"]

# PLAY WEIGHT (VECTORIZED)
"""
Assigns a weight to a (user, track) play record showing how much the user likes the track.
 - Tracks that are frequently skipped early get a weight of 0.
 - Otherwise, weight increases with the percentage of meaningful listens and 
    the log of total play count (to dampen the reduced impact of excessive replays).
"""
# log1p dampens obsessive replays — 10 plays shouldn't outweigh 2 plays by 5x
play_stats["weight"] = play_stats["meaningful_pct"] * np.log1p(play_stats["n_plays"])

# Zero out tracks the user consistently bails on early
dislike_mask = (
    (play_stats["skip_rate"] > MIN_SKIP_RATE_DISLIKE)
    & (play_stats["avg_ms"] < MAX_MS_DISLIKE)
)

play_stats.loc[dislike_mask, "weight"] = 0.0
play_stats.to_csv(DATA_PROC / "play_stats.csv", index=False)

# Map users to row indices for the sparse matrix
user_to_idx = {u: i for i, u in enumerate(USERS)}
play_stats["user_idx"] = play_stats["user"].map(user_to_idx)

# Build a sparse user × track weight matrix
# Rows = users, Columns = tracks, Values = play weight
user_track_matrix = csr_matrix(
    (play_stats["weight"].values, (play_stats["user_idx"].values, play_stats["track_id"].values)),
    shape=(len(USERS), audio_matrix_norm.shape[0])
)

# Precompute sum of weights per user to normalize later
weight_sum = np.array(user_track_matrix.sum(axis=1)).flatten()

# LONG-TERM AUDIO PROFILES
"""
Builds a single vector representing each user's overall audio taste.
- Weighted average of all tracks the user listened to.
- Tracks with higher engagement contribute more to the profile.
"""
# Matrix multiply gives weighted sum; divide by weight_sum to get weighted average
_ltp_matrix = user_track_matrix @ audio_matrix_norm
_ltp_matrix = _ltp_matrix / (weight_sum[:, None] + 1e-9) # +1e-9 avoids div-by-zero
long_term_profiles = {u: _ltp_matrix[user_to_idx[u]] for u in USERS}

# TAG DISTRIBUTIONS PER USER
"""
Builds weighted tag profiles for all users in one pass.
- Captures which genres and descriptors are characteristic of each user's taste.
- Weighted by play engagement.
"""
_utp_matrix = user_track_matrix @ tag_matrix
_utp_matrix = _utp_matrix / (weight_sum[:, None] + 1e-9)
user_tag_profiles = {u: _utp_matrix[user_to_idx[u]] for u in USERS}

# MOOD CLUSTERS PER USER
"""
Identifies mood-based clusters in a user's listening history.
- Uses a subset of audio features (MOOD_FEATURES).
- Weighted KMeans clustering per user.
- Returns cluster centroids, labels, track indices, and weighted tag distributions per cluster.
"""
# Extract mood features and fill NaNs with 0 before scaling
mood_feat_idx = [AUDIO_FEATURES.index(f) for f in MOOD_FEATURES]
mood_matrix = np.nan_to_num(audio_matrix[:, mood_feat_idx], nan=0.0)

# Scale to [0,1]
mood_scaler = MinMaxScaler()
mood_matrix_norm = mood_scaler.fit_transform(mood_matrix)

def build_mood_clusters(user: str):
    u_idx = user_to_idx[user]
    # .indices/.data give the non-zero columns and values from the sparse row
    idx = user_track_matrix[u_idx].indices
    w = user_track_matrix[u_idx].data

    # Handle users with no meaningful plays
    if len(idx) == 0:
        return np.empty((0, len(MOOD_FEATURES))), np.array([]), np.array([]), {}

    X = mood_matrix_norm[idx]
    k = min(N_MOOD_CLUSTERS, len(idx))
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X, sample_weight=w)

    tag_dist = {}
    for c in range(k):
        mask = labels == c
        c_idx, c_w = idx[mask], w[mask]
        if len(c_idx) == 0:
            tag_dist[c] = np.zeros(tag_matrix.shape[1])
        else:
            ws = np.array((tag_matrix[c_idx] * c_w[:, None]).sum(axis=0)).flatten()
            tag_dist[c] = ws / (c_w.sum() + 1e-9)
    return km.cluster_centers_, labels, idx, tag_dist

print("Building mood clusters...")
mood_clusters = {u: build_mood_clusters(u) for u in USERS}

# Save fitted mood scaler for later use
joblib.dump(mood_scaler, MODELS_DIR / "mood_scaler.joblib")

# SAVE LONG-TERM AUDIO PROFILES
# Each row = user
# Each column = audio feature
pd.DataFrame([long_term_profiles[u] for u in USERS], columns=AUDIO_FEATURES) \
    .assign(user=USERS) \
    .to_csv(DATA_PROC / "user_profiles.csv", index=False)

# SAVE TAG PROFILES
# Each row = user
# Each column = tag
pd.DataFrame([user_tag_profiles[u] for u in USERS], columns=tag_vocab) \
    .assign(user=USERS) \
    .to_csv(DATA_PROC / "user_tag_profiles.csv", index=False)

# SAVE MOOD CLUSTERS PER USER
# One CSV per user
# Rows = cluster centroids
for u in USERS:
    centroids, *_ = mood_clusters[u]
    pd.DataFrame(centroids, columns=MOOD_FEATURES) \
        .rename_axis("cluster") \
        .reset_index() \
        .assign(user=u) \
        .to_csv(DATA_PROC / f"mood_clusters_{u}.csv", index=False)
    
# USER-USER SIMILARITY
"""
Computes pairwise user similarity using the tag profiles.
Uses cosine similarity between all users.
"""
user_sim_df = pd.DataFrame(
    cosine_similarity(np.stack([user_tag_profiles[u] for u in USERS])),
    index=USERS, columns=USERS
)

print("\nUser profiles saved -> data/processed/")
print("mood_scaler.joblib -> models/")
print("\n", user_sim_df.round(3))


Building mood clusters...

User profiles saved -> data/processed/
mood_scaler.joblib -> models/

            andy  dishita   riya  priyanka
andy      1.000    0.835  0.534     0.906
dishita   0.835    1.000  0.628     0.818
riya      0.534    0.628  1.000     0.741
priyanka  0.906    0.818  0.741     1.000


In [9]:
# CANDIDATE GENERATION

"""
Finds tracks most similar to the user's taste using audio feature similarity. 
Queries against both the user's long-term profile and each of their mood centroids, returning 
the union of top matches. 
Tracks the user consistently skips early are excluded.
"""
def content_knn_candidates(user: str) -> set:
    # Tracks the user frequently skips early. Treated as a dislike
    bad = set(
        play_stats[
            (play_stats["user"] == user) & (play_stats["skip_rate"] > MIN_SKIP_RATE_DISLIKE)
        ]["track_id"]
    )

    base = long_term_profiles[user].copy()
    queries = [base]
    centroids, *_ = mood_clusters[user]

    # Build one query vector per mood centroid by splicing centroid values 
    # into the user's long-term profile
    for c in centroids:
        v = base.copy()
        
        for i, fi in enumerate(mood_feat_idx):
            v[fi] = c[i]
            
        queries.append(v)

    candidates = set()
    for qv in queries:
        sims = cosine_similarity(qv.reshape(1, -1), audio_matrix_norm)[0]
        top_idx = np.argsort(sims)[::-1][:TOP_N_CONTENT]
        candidates.update(int(i) for i in top_idx if i not in bad)

    return candidates

"""
Finds tracks that match the user's top genre/mood tags.
Also finds tracks that have been played by at least one person in the group. 
Expands the candidate pool with genre-relevant tracks validated by how much a track has been 
played by other users.
"""
def tag_expansion_candidates(user: str) -> set:
    # Column indices of the user's top-k tags in the TF-IDF vocab
    top_tag_idx = np.argsort(user_tag_profiles[user])[::-1][:TOP_K_TAGS]
    # Sum TF-IDF scores across top tags to rank every track by genre relevance
    tag_scores = np.array(tag_matrix[:, top_tag_idx].sum(axis=1)).flatten()

    group_relevant = set(
        play_stats.groupby("track_id")["user"]
        .nunique()
        .loc[lambda x: x >= 1]
        .index
    )

    candidates = set()
    for tid in np.argsort(tag_scores)[::-1]:
        if int(tid) in group_relevant:
            candidates.add(int(tid))
        if len(candidates) >= TOP_N_TAG:
            break

    return candidates

"""
Finds tracks that similar users (neighbors) listened to frequently, but the target user has not yet 
heard. 
Neighbors are weighted by their tag-based similarity to the target user.
(Tracks from more similar users in the group contribute more). 
(THE Collaborative filtering).
"""
def neighbor_expansion_candidates(user: str) -> set:
    neighbors = [u for u in USERS if u != user]
    sim_scores = {n: float(user_sim_df.loc[user, n]) for n in neighbors}

    # Tracks the user has already heard enough to be considered "known"
    u_played = set(play_stats[
        (play_stats["user"] == user) & (play_stats["n_plays"] > NEIGHBOR_PLAY_THRESHOLD)
    ]["track_id"])

    nbr_score: dict[int, float] = {}
    for nbr in neighbors:
        good = play_stats[
            (play_stats["user"] == nbr)
            & (play_stats["avg_ms"] >= MIN_NEIGHBOR_MS)
            & (play_stats["skip_rate"] <= MAX_NEIGHBOR_SKIP)
        ]

        for _, row in good.iterrows():
            tid = int(row["track_id"])
            if tid not in u_played:
                # Weight neighbor's signal by how similar they are to the target user
                nbr_score[tid] = (nbr_score.get(tid, 0) + sim_scores[nbr] * row["meaningful_pct"])

    ranked = sorted(nbr_score, key=nbr_score.get, reverse=True)
    return set(ranked[:TOP_N_NEIGHBOR])

"""
Merges all three candidate sources (content k-NN, tag expansion, neighbor expansion).
Computes a pre-score for each track combining:
 - Content similarity
 - Tag similarity
 - Group popularity
  - Neighbor signal
Returns the top FINAL_CANDIDATE_K tracks sorted by pre-score.
"""
def build_candidates(user: str) -> pd.DataFrame:
    content_cands = content_knn_candidates(user)
    tag_cands = tag_expansion_candidates(user)
    neighbor_cands = neighbor_expansion_candidates(user)
    all_cands = content_cands | tag_cands | neighbor_cands

    # Score every candidate against the full track matrix in one pass
    content_sims = cosine_similarity(long_term_profiles[user].reshape(1, -1),
                                     audio_matrix_norm)[0]

    tag_sims = cosine_similarity(user_tag_profiles[user].reshape(1, -1),
                                 tag_matrix)[0]
    
    # Fraction of the group (0–1) who have played this track
    group_pop = (
        play_stats.groupby("track_id")["user"]
        .nunique()
        .div(len(USERS))
        .to_dict()
    )

    # Re-compute neighbor signal across all tracks (not just neighbor_cands)
    # so every candidate gets a meaningful neighbor score, not just 0 or signal
    neighbors = [u for u in USERS if u != user]
    sim_scores = {n: float(user_sim_df.loc[user, n]) for n in neighbors}
    nbr_signal: dict[int, float] = {}

    for nbr in neighbors:
        good = play_stats[(play_stats["user"] == nbr) & (play_stats["avg_ms"] >= MIN_NEIGHBOR_MS)]
        for _, row in good.iterrows():
            tid = int(row["track_id"])
            nbr_signal[tid] = (nbr_signal.get(tid, 0) + sim_scores[nbr] * row["meaningful_pct"])

    # Normalize neighbor signal to [0, 1] so it's on the same scale as other scores
    if nbr_signal:
        max_nbr = max(nbr_signal.values())
        nbr_signal = {k: v / max_nbr for k, v in nbr_signal.items()}

    rows = []
    for tid in all_cands:
        cs = float(content_sims[tid])
        ts = float(tag_sims[tid])
        gp = float(group_pop.get(tid, 0))
        ns = float(nbr_signal.get(tid, 0))
        rows.append({
            "track_id": tid,
            "pre_score": (
                PRE_SCORE_CONTENT_W * cs
                + PRE_SCORE_TAG_W * ts
                + PRE_SCORE_GROUP_W * gp
                + PRE_SCORE_NEIGHBOR_W * ns
            ),
            "content_sim": cs,
            "tag_sim": ts,
            "group_pop": gp,
            "neighbor_score": ns,
            "from_content": tid in content_cands,
            "from_tag": tid in tag_cands,
            "from_neighbor": tid in neighbor_cands,
        })

    df = (
        pd.DataFrame(rows)
        .sort_values("pre_score", ascending=False)
        .head(FINAL_CANDIDATE_K)
        .reset_index(drop=True)
        .merge(
            track_table[["track_id", "track_name", "artist_name"]],
            on="track_id",
            how="left",
        )
    )

    return df

candidate_pools = {}
for user in USERS:
    candidate_pools[user] = build_candidates(user)
    candidate_pools[user].to_csv(DATA_PROC / f"candidates_{user}.csv", index=False)
    print(f"{user:10s}: {len(candidate_pools[user]):,} candidates")

print("\nCandidate generation complete.")

andy      : 1,000 candidates
dishita   : 1,000 candidates
riya      : 1,000 candidates
priyanka  : 1,000 candidates

Candidate generation complete.


In [10]:
#SCORING

# Helper function to help scale values to [0, 1] for fair combination into final score (that way one
# feature with a wider range doesn't dominate the final score)
def _minmax(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)

"""
This function determines the active session mood by looking at the user's most recently played tracks and seeing which
mood cluster they belong to.
"""
def _get_session_centroid(user, last_n=10):
    centroids, labels, idx, tag_dist = mood_clusters[user]

    if len(centroids) == 0:
        return None, None

    # get the user's most recently played track ids
    recent_track_ids = (
        events[events["user"] == user]
        .sort_values("ts", ascending=False)
        .drop_duplicates("track_id")
        .head(last_n)["track_id"]
        .values
    )

    # find which of those tracks exist in the user's mood cluster history
    recent_in_history = [t for t in recent_track_ids if t in idx]

    if not recent_in_history:
        # no overlap — fall back to first centroid
        return centroids[0], tag_dist[0]

    # for each recent track that's in the history, look up which cluster it belongs to
    idx_list    = list(idx)
    cluster_ids = [labels[idx_list.index(t)] for t in recent_in_history]

    # whichever cluster appears most among recent tracks is the active session mood
    active_cluster = max(set(cluster_ids), key=cluster_ids.count)
    return centroids[active_cluster], tag_dist[active_cluster]

"""
Extracts features for a given user and candidate tracks, including:
- Content similarity to the user's long-term profile
- Tag similarity to the user's tag profile
- Mood similarity to the active session mood centroid
- Cluster tag similarity to the active session mood's tag distribution
- Tag Jaccard similarity between the track's top tags and the user's top tags
- Group popularity (fraction of users who have played the track)
- Group average listen duration and skip rate
- Neighbor signal (weighted engagement from similar users)

Returns a DataFrame with one row per candidate track and all the extracted features.
"""

def _content_features(user, track_ids, session_mood_centroid, session_tag_dist, candidates_df):
    pre = candidates_df.set_index("track_id")

    if session_mood_centroid is not None:
        mood_query = np.zeros((1, audio_matrix_norm.shape[1]))
        for i, fi in enumerate(mood_feat_idx):
            mood_query[0, fi] = session_mood_centroid[i]
        all_mood_sims = cosine_similarity(mood_query, audio_matrix_norm)[0]
    else:
        all_mood_sims = np.zeros(audio_matrix_norm.shape[0])

    # cluster tag similarity — how well does this track match the active mood's tag profile
    # different from tag_cosine which uses the user's overall long-term tag profile
    if session_tag_dist is not None:
        all_cluster_tag_sims = cosine_similarity(session_tag_dist.reshape(1, -1), tag_matrix)[0]
    else:
        all_cluster_tag_sims = np.zeros(tag_matrix.shape[0])

    top_user_tag_idx_ranked = np.argsort(user_tag_profiles[user])[::-1][:3]
    top_user_tag_idx        = set(top_user_tag_idx_ranked)

    rows = []
    for tid in track_ids:
        track_tag_vec     = np.asarray(tag_matrix[tid]).flatten()
        top_track_tag_idx = set(np.argsort(track_tag_vec)[::-1][:5])
        intersection      = top_track_tag_idx & top_user_tag_idx
        union             = top_track_tag_idx | top_user_tag_idx

        rows.append({
            "track_id":        tid,
            "long_term_sim":   float(pre.loc[tid, "content_sim"]),        # from candidates_df
            "tag_cosine":      float(pre.loc[tid, "tag_sim"]),             # from candidates_df
            "mood_sim":        float(all_mood_sims[tid]),                  # new
            "cluster_tag_sim": float(all_cluster_tag_sims[tid]),           # new
            "tag_jaccard":     float(len(intersection) / len(union)) if union else 0.0,  # new
        })

    return pd.DataFrame(rows)

"""
Extracts collaborative features for candidate tracks, including:
- Group listener count (fraction of users who have played the track)
- Group average listen duration and skip rate
- Neighbor signal (weighted engagement from similar users)

Returns a DataFrame with one row per candidate track and all the extracted collaborative features.
"""
def _collab_features(user, track_ids, candidates_df):
    pre       = candidates_df.set_index("track_id")
    neighbors = [u for u in USERS if u != user]

    grp = (
        play_stats.groupby("track_id")
        .agg(
            group_avg_ms   =("avg_ms",    "mean"),
            group_skip_rate=("skip_rate", "mean"),
        )
        .to_dict("index")
    )

    nbr_weighted_ms = {}
    nbr_weight_sum  = {}
    for nbr in neighbors:
        w = float(user_sim_df.loc[user, nbr])
        for _, row in play_stats[play_stats["user"] == nbr].iterrows():
            tid = int(row["track_id"])
            nbr_weighted_ms[tid] = nbr_weighted_ms.get(tid, 0.0) + w * row["avg_ms"]
            nbr_weight_sum[tid]  = nbr_weight_sum.get(tid, 0.0)  + w

    rows = []
    for tid in track_ids:
        g  = grp.get(tid, {})
        ws = nbr_weight_sum.get(tid, 0.0)
        rows.append({
            "track_id":             tid,
            "group_listener_count": float(pre.loc[tid, "group_pop"]) * len(USERS),  # from candidates_df
            "group_avg_ms":         float(g.get("group_avg_ms",    0)),              # new
            "group_skip_rate":      float(g.get("group_skip_rate", 1)),              # new
            "weighted_nbr_avg_ms":  float(nbr_weighted_ms.get(tid, 0.0) / ws) if ws > 0 else 0.0,  # new
        })

    return pd.DataFrame(rows)

"""
Builds training labels for candidate tracks based on the user's play history.
- Liked (1): avg_ms > liked_ms threshold AND skip_rate < 20%
- Disliked (0): skip_rate > skip_thresh
- Neutral/Unknown (-1): everything else (including tracks the user hasn't played)

Returns a numpy array of labels corresponding to the input track_ids, which can be used for training a model
to learn feature weights.
"""

def _session_features(user, track_ids, last_n=10):
    recent_track_ids = (
        events[events["user"] == user]
        .sort_values("ts", ascending=False)
        .drop_duplicates("track_id")
        .head(last_n)["track_id"]
    )

    recent_stats = play_stats[
        (play_stats["user"] == user) &
        (play_stats["track_id"].isin(recent_track_ids))
    ]

    recent_skip_pct = float(recent_stats["skip_rate"].mean()) if len(recent_stats) else 0.5

    recent_meta        = track_table[track_table["track_id"].isin(recent_track_ids)]
    recent_avg_valence = float(recent_meta["valence"].mean()) if len(recent_meta) else 0.5
    recent_avg_energy  = float(recent_meta["energy"].mean())  if len(recent_meta) else 0.5

    rows = []
    for tid in track_ids:
        t            = track_table[track_table["track_id"] == tid]
        cand_valence = float(t["valence"].iloc[0]) if len(t) else 0.5
        cand_energy  = float(t["energy"].iloc[0])  if len(t) else 0.5

        rows.append({
            "track_id":           tid,
            "recent_skip_pct":    recent_skip_pct,
            "recent_avg_valence": recent_avg_valence,
            "recent_avg_energy":  recent_avg_energy,
            "valence_delta":      abs(cand_valence - recent_avg_valence),
            "energy_delta":       abs(cand_energy  - recent_avg_energy),
        })

    return pd.DataFrame(rows)

"""
Extracts contextual features for candidate tracks, such as time of day and platform.
- Time of day is bucketed into morning, afternoon, evening, night based on typical listening
    patterns.
- Platform is categorized into mobile vs desktop.

Returns a DataFrame with one row per candidate track and binary features indicating the presence of each context
"""

def _context_features(user, track_ids, time_of_day_bucket=None, platform=None):
    tod_buckets  = ["morning", "afternoon", "evening", "night"]
    plat_buckets = ["mobile", "desktop"]

    rows = []
    for tid in track_ids:
        row = {"track_id": tid}
        for b in tod_buckets:
            row[f"tod_{b}"]      = int(time_of_day_bucket == b)
        for p in plat_buckets:
            row[f"platform_{p}"] = int(platform == p)
        rows.append(row)

    return pd.DataFrame(rows)


"""
Computes a hybrid content score based on multiple similarity metrics.
- long_term_sim: cosine similarity to user's long-term audio profile
- tag_sim: cosine similarity to user's tag profile
- mood_sim: cosine similarity to active session mood centroid
- cluster_tag_sim: cosine similarity to active session mood's tag distribution
- tag_jaccard: Jaccard similarity between track's top tags and user's top tags
Each component is normalized to [0, 1] and combined with specific weights to produce a final content score.

Returns a numpy array of content scores corresponding to the input feature DataFrame.
"""
def _compute_content_score(feat_df):
    lt_norm          = _minmax(feat_df["long_term_sim"].values)
    mood_norm        = _minmax(feat_df["mood_sim"].values)
    tag_norm         = _minmax(feat_df["tag_cosine"].values)
    cluster_tag_norm = _minmax(feat_df["cluster_tag_sim"].values)
    jaccard          = feat_df["tag_jaccard"].values           # already [0, 1]

    # cluster_tag_sim and mood_sim together represent the session mood signal
    # weighted slightly lower than long-term signals since they're more volatile
    s_c = (
        0.30 * lt_norm
        + 0.25 * tag_norm
        + 0.20 * mood_norm
        + 0.15 * cluster_tag_norm
        + 0.10 * jaccard
    )
    return s_c.clip(0, 1)

"""
Computes a collaborative score based on group engagement metrics.
- group_listener_count: how many users in the group have played the track
- group_avg_ms: average listen duration across the group
- group_skip_rate: average skip rate across the group
- weighted_nbr_avg_ms: neighbor signal based on how much similar users have engaged with the track
Each component is normalized to [0, 1] and combined with specific weights to produce a final collaborative score.

Returns a numpy array of collaborative scores corresponding to the input feature DataFrame and total number of users (for normalization).
"""
def _compute_collab_score(feat_df, n_users):
    pop_norm  = _minmax(feat_df["group_listener_count"].values / max(n_users, 1))
    ms_norm   = _minmax(feat_df["group_avg_ms"].values)
    skip_norm = 1.0 - _minmax(feat_df["group_skip_rate"].values)
    ms_nbr    = _minmax(feat_df["weighted_nbr_avg_ms"].values)

    s_cf = (
        0.30 * pop_norm
        + 0.25 * ms_norm
        + 0.20 * skip_norm
        + 0.25 * ms_nbr
    )
    return s_cf.clip(0, 1)

"""
OPTIONAL: Builds training labels for candidate tracks based on the user's play history.
- Liked (1): avg_ms > liked_ms threshold AND skip_rate < 20%
- Disliked (0): skip_rate > skip_thresh
- Neutral/Unknown (-1): everything else (including tracks the user hasn't played)

Returns a numpy array of labels corresponding to the input track_ids, which can be used for training a model to learn feature weights.
"""
def _build_labels(user, track_ids, liked_ms=45_000, skip_thresh=0.60):
    u_stats = play_stats[play_stats["user"] == user].set_index("track_id")
    labels  = []
    for tid in track_ids:
        if tid in u_stats.index:
            r = u_stats.loc[tid]
            if r["avg_ms"] > liked_ms and r["skip_rate"] < 0.20:
                labels.append(1)
            elif r["skip_rate"] > skip_thresh:
                labels.append(0)
            else:
                labels.append(-1)
        else:
            labels.append(-1)
    return np.array(labels)

"""
OPTIONAL: Learns optimal weights for combining content and collaborative scores using logistic regression.
- Takes the content and collaborative scores for candidate tracks, along with the binary labels of whether the
user liked or disliked the track.
- Trains a logistic regression model to predict the labels from the scores.
- Extracts the absolute values of the learned coefficients to determine the relative importance of content vs collab features.
- Normalizes the weights to sum to 1 for interpretability.

Returns the learned weights for content and collaborative scores, which can be used to combine them into a final hybrid score.
"""
def _learn_weights(s_c, s_cf, labels):
    X      = np.column_stack([s_c, s_cf])
    scaler = StandardScaler()
    clf    = LogisticRegression(max_iter=200)
    clf.fit(scaler.fit_transform(X), labels)
    coefs     = np.abs(clf.coef_[0])
    w_c, w_cf = coefs / (coefs.sum() + 1e-9)
    return float(w_c), float(w_cf)

"""
Combines content and collaborative features into a final hybrid score for candidate tracks.
- Extracts content features, collaborative features, session features, and contextual features for the candidate tracks
- Computes separate content and collaborative scores using the defined scoring functions
- Optionally learns weights for combining the scores based on the user's play history
- Merges the scores with the original candidate information and track metadata
- Sorts the candidates by the final hybrid score and returns the top K tracks with all relevant information for analysis

Returns a DataFrame with the top K scored tracks for the user, including the hybrid score, individual feature scores,
source indicators, and track metadata.
"""
def hybrid_score(
    user,
    candidates_df,
    session_mood_centroid = None,
    session_tag_dist      = None,
    time_of_day_bucket    = None,
    platform              = None,
    use_learned_weights   = False, # If we set this to true, we can try the methods that learn weights from user history,but for now its just set to 80/20 fixed weights
    top_k                 = FINAL_TOP_K,
):
    track_ids = candidates_df["track_id"].tolist()

    content_feat = _content_features(user, track_ids, session_mood_centroid, session_tag_dist, candidates_df)
    collab_feat  = _collab_features(user, track_ids, candidates_df)
    session_feat = _session_features(user, track_ids)
    context_feat = _context_features(user, track_ids, time_of_day_bucket, platform)

    feat_df = (
        content_feat
        .merge(collab_feat,  on="track_id")
        .merge(session_feat, on="track_id")
        .merge(context_feat, on="track_id")
    )

    s_c  = _compute_content_score(feat_df)
    s_cf = _compute_collab_score(feat_df, n_users=len(USERS))

    feat_df["s_content"] = s_c
    feat_df["s_collab"]  = s_cf

    if use_learned_weights:
        labels = _build_labels(user, track_ids)
        mask   = labels != -1
        if mask.sum() >= 10:
            w_c, w_cf = _learn_weights(s_c[mask], s_cf[mask], labels[mask])
        else:
            w_c, w_cf = W_CONTENT, W_COLLAB
    else:
        w_c, w_cf = W_CONTENT, W_COLLAB

    feat_df["w_content"]    = w_c
    feat_df["w_collab"]     = w_cf
    feat_df["hybrid_score"] = w_c * s_c + w_cf * s_cf

    source_cols = candidates_df[["track_id", "pre_score", "from_content", "from_tag", "from_neighbor"]]

    result = (
        feat_df
        .merge(source_cols, on="track_id", how="left")
        .merge(
            track_table[["track_id", "track_name", "artist_name", "artist_id"]],
            on="track_id",
            how="left",
        )
        .sort_values("hybrid_score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    lead_cols = [
        "track_id", "track_name", "artist_name", "artist_id",
        "hybrid_score", "s_content", "s_collab", "w_content", "w_collab",
        "pre_score", "from_content", "from_tag", "from_neighbor",
    ]
    feat_cols = [c for c in result.columns if c not in lead_cols]
    return result[lead_cols + feat_cols]


"""
For each user, builds the final scored pool of tracks by applying the hybrid scoring function to the candidate pool.
Saves the scored pools to CSV for analysis and debugging.
Prints out the top 5 scored tracks for each user along with their feature scores and source indicators to understand why
they were recommended.
"""
scored_pools = {}
for user in USERS:
    session_centroid, session_tag_dist = _get_session_centroid(user)

    scored_pools[user] = hybrid_score(
        user                  = user,
        candidates_df         = candidate_pools[user],
        session_mood_centroid = session_centroid,
        session_tag_dist      = session_tag_dist,
        time_of_day_bucket    = None,
        platform              = None,
        use_learned_weights   = False,
        top_k                 = FINAL_TOP_K,
    )

    scored_pools[user].to_csv(DATA_PROC / f"hybrid_scored_{user}.csv", index=False) 

    print(f"{user}  w_c={scored_pools[user]['w_content'].iloc[0]:.2f}  "
          f"w_cf={scored_pools[user]['w_collab'].iloc[0]:.2f}  "
          f"pool={len(scored_pools[user])}")
    print(
        scored_pools[user][
            ["track_name", "artist_name", "hybrid_score", "pre_score",
             "from_content", "from_tag", "from_neighbor"]
        ].head(5).to_string(index=False)
    )
    print()

print("Hybrid scoring complete.")

andy  w_c=0.80  w_cf=0.20  pool=150
                       track_name       artist_name  hybrid_score  pre_score  from_content  from_tag  from_neighbor
         Running Through The Rain           YUGYEOM      0.784124   0.708342          True      True          False
                        SUMMERIDE          Jay Park      0.770534   0.699218          True     False          False
                  Didn't Cha Know       Erykah Badu      0.766561   0.871330         False      True          False
Frontin' (feat. JAY-Z) - Club Mix Pharrell Williams      0.758643   0.895538         False      True          False
                       Can't Wait          Doja Cat      0.753219   0.729839          True     False          False

dishita  w_c=0.80  w_cf=0.20  pool=150
              track_name artist_name  hybrid_score  pre_score  from_content  from_tag  from_neighbor
        act ii: date @ 8       4batz      0.740810   0.739676          True     False          False
                  Jammin  

In [11]:
# RE-RANKING: MEAN AVERAGE PRECISION
# Uses each user's historical play data as ground truth to find the optimal
# content/collab weight split, then re-scores and re-ranks the 150-track scored pool.
# The result is a new ranked list that is provably better ordered w.r.t. the user's
# known preferences than the fixed 80/20 hybrid score alone.

# TUNEABLE PARAMETERS FOR RE-RANKING
MAP_WEIGHT_STEPS  = 20       # number of steps when sweeping content/collab weights
MAP_LIKED_MS      = 45_000   # ms threshold to consider a track "liked" (ground truth positive)
MAP_SKIP_THRESH   = 0.20     # skip rate must be below this to count as liked
MAP_MIN_POSITIVES = 3        # minimum # of ground-truth positives needed to trust the MAP signal;
                             # if fewer exist in the pool, auto-relax thresholds before giving up

# FALLBACK THRESHOLD LADDER
# Tried in order when the primary thresholds yield too few positives in the pool.
# Each tuple is (liked_ms, skip_thresh). Progressively looser until we hit MAP_MIN_POSITIVES
# or exhaust all options, at which point defaults are kept.
MAP_FALLBACK_THRESHOLDS = [
    (30_000, 0.30),   # tier 1: shorter listen, slightly more skips allowed
    (20_000, 0.40),   # tier 2: near-minimum meaningful play
    (10_000, 0.50),   # tier 3: very loose — any meaningful engagement at all
]

# WEIGHT GRID: evenly spaced pairs (w_content, w_collab) that sum to 1
# e.g. [(1.0, 0.0), (0.95, 0.05), ..., (0.0, 1.0)]
_weight_grid = [
    (round(w, 2), round(1.0 - w, 2))
    for w in np.linspace(0.0, 1.0, MAP_WEIGHT_STEPS + 1)
]

"""
Builds a binary relevance set for a given user from their full play history.
A track is considered relevant (ground truth positive) if:
  - The user has played it at least once
  - Their average listen duration exceeds liked_ms
  - Their skip rate is below skip_thresh
- liked_ms    : minimum average ms listened to count as liked (default: MAP_LIKED_MS)
- skip_thresh : maximum skip rate to count as liked (default: MAP_SKIP_THRESH)
Returns a set of track_ids the user genuinely liked under the given thresholds.
"""
def _build_relevant_set(user: str, liked_ms: int = MAP_LIKED_MS, skip_thresh: float = MAP_SKIP_THRESH) -> set:
    u_stats = play_stats[play_stats["user"] == user]
    relevant = u_stats[
        (u_stats["avg_ms"]    > liked_ms)    &
        (u_stats["skip_rate"] < skip_thresh)
    ]["track_id"]
    return set(relevant.astype(int))

"""
Computes Average Precision (AP) for a single ranked list given a set of relevant track ids.
AP = mean of precision@k values taken only at positions where a relevant track appears.
- ranked_ids : list of track_ids in ranked order (best first)
- relevant   : set of track_ids considered ground-truth positives
Returns AP as a float in [0, 1]. Returns 0.0 if there are no relevant tracks in the list.
"""
def _average_precision(ranked_ids: list, relevant: set) -> float:
    if not relevant:
        return 0.0
    hits        = 0
    score_sum   = 0.0
    for rank, tid in enumerate(ranked_ids, start=1):
        if tid in relevant:
            hits      += 1
            score_sum += hits / rank      # precision@rank, only counted at hit positions
    if hits == 0:
        return 0.0
    return score_sum / len(relevant)      # normalise by total number of relevant tracks

"""
Runs a weight grid sweep over (w_content, w_collab) pairs for a given relevant set.
Returns the best (w_content, w_collab, ap) found across all weight combinations.
Helper for _find_best_weights — separated so it can be called per threshold tier.
"""
def _sweep_weights(scored_df: pd.DataFrame, relevant_set: set) -> tuple:
    s_c  = scored_df["s_content"].values
    s_cf = scored_df["s_collab"].values
    ids  = scored_df["track_id"].astype(int).tolist()

    best_ap, best_wc, best_wcf = -1.0, W_CONTENT, W_COLLAB

    for w_c, w_cf in _weight_grid:
        trial_scores = w_c * s_c + w_cf * s_cf
        ranked_order = np.argsort(trial_scores)[::-1]
        ranked_ids   = [ids[i] for i in ranked_order]
        ap           = _average_precision(ranked_ids, relevant_set)
        if ap > best_ap:
            best_ap, best_wc, best_wcf = ap, w_c, w_cf

    return (best_wc, best_wcf, best_ap)

"""
Sweeps over the weight grid to find the (w_content, w_collab) pair that maximises
Average Precision for this user on their current scored pool.
- user      : user name string
- scored_df : the 150-row hybrid_score output for this user
First tries the primary thresholds (MAP_LIKED_MS, MAP_SKIP_THRESH).
If fewer than MAP_MIN_POSITIVES exist in the pool at the primary thresholds, walks
MAP_FALLBACK_THRESHOLDS in order — progressively looser — until enough positives are
found or all tiers are exhausted.
At each tier, only accepts the result if the best AP found is strictly greater than the
AP of the original hybrid_score ordering under those same thresholds. If no tier
produces a genuine improvement, returns (W_CONTENT, W_COLLAB, 0.0) to keep defaults.
Returns (best_w_content, best_w_collab, best_ap, relevant_set_used, tier_label).
"""
def _find_best_weights(user: str, scored_df: pd.DataFrame) -> tuple:
    pool_ids = set(scored_df["track_id"].astype(int))

    # Build threshold ladder: primary first, then fallbacks
    all_tiers = [(MAP_LIKED_MS, MAP_SKIP_THRESH, "primary")] + [
        (ms, sk, f"fallback-tier{i+1}")
        for i, (ms, sk) in enumerate(MAP_FALLBACK_THRESHOLDS)
    ]

    for liked_ms, skip_thresh, tier_label in all_tiers:
        relevant_set   = _build_relevant_set(user, liked_ms, skip_thresh)
        pool_positives = relevant_set & pool_ids

        if len(pool_positives) < MAP_MIN_POSITIVES:
            continue   # not enough signal at this tier — try next

        # Baseline AP: original hybrid_score ordering under this tier's relevant set
        original_ranked = scored_df["track_id"].astype(int).tolist()
        ap_before       = _average_precision(original_ranked, relevant_set)

        best_wc, best_wcf, best_ap = _sweep_weights(scored_df, relevant_set)

        # Only accept if we actually beat the baseline — otherwise keep looking
        if best_ap > ap_before:
            return (best_wc, best_wcf, best_ap, relevant_set, tier_label)

    # Exhausted all tiers without a genuine improvement — keep defaults
    # Return the primary relevant set for AP reporting, even if it has few positives
    fallback_relevant = _build_relevant_set(user)
    return (W_CONTENT, W_COLLAB, 0.0, fallback_relevant, "defaults")

"""
Applies MAP-optimised weights to re-score and re-rank a user's scored pool.
- Calls _find_best_weights which automatically walks the threshold ladder until
  enough ground-truth positives exist in the pool to produce a genuine AP improvement.
- Re-computes map_score = best_w_content * s_content + best_w_collab * s_collab.
- Sorts by map_score descending to produce the final re-ranked playlist.
- Attaches diagnostics: best weights found, AP before/after, and which threshold
  tier produced the result.
Returns a DataFrame in the same schema as scored_pools[user], with added columns:
  map_score, map_w_content, map_w_collab, ap_before, ap_after, map_tier.
"""
def map_rerank(user: str, scored_df: pd.DataFrame) -> pd.DataFrame:
    best_wc, best_wcf, best_ap, relevant_set, tier_label = _find_best_weights(user, scored_df)

    # ap_before is always computed under the same relevant set that was actually used,
    # so ap_before and ap_after are directly comparable.
    original_ranked = scored_df["track_id"].astype(int).tolist()
    ap_before       = _average_precision(original_ranked, relevant_set)

    # Only call it an improvement if best_ap is strictly greater — not equal, not zero
    genuine_improvement = best_ap > ap_before and best_ap > 0.0

    reranked_df = scored_df.copy()
    reranked_df["map_score"]     = best_wc * reranked_df["s_content"] + best_wcf * reranked_df["s_collab"]
    reranked_df["map_w_content"] = best_wc
    reranked_df["map_w_collab"]  = best_wcf
    reranked_df["ap_before"]     = ap_before
    reranked_df["ap_after"]      = best_ap if genuine_improvement else ap_before
    reranked_df["map_tier"]      = tier_label
    reranked_df["map_improved"]  = genuine_improvement

    reranked_df = (
        reranked_df
        .sort_values("map_score", ascending=False)
        .reset_index(drop=True)
    )

    return reranked_df

# RUN MAP RE-RANKING FOR ALL USERS
print("Running MAP re-ranking...\n")
reranked_pools = {}

for user in USERS:
    reranked_pools[user] = map_rerank(user, scored_pools[user])
    reranked_pools[user].to_csv(DATA_PROC / f"map_reranked_{user}.csv", index=False)

    r        = reranked_pools[user]
    wc       = r["map_w_content"].iloc[0]
    wcf      = r["map_w_collab"].iloc[0]
    apb      = r["ap_before"].iloc[0]
    apa      = r["ap_after"].iloc[0]
    tier     = r["map_tier"].iloc[0]
    improved = r["map_improved"].iloc[0]

    if improved:
        status = "improved"
    elif tier == "defaults":
        status = "no signal in pool — defaults kept"
    else:
        status = "weights adjusted but AP unchanged — marginal re-rank only"

    print(f"{user}")
    print(f"  tier    : {tier}")
    print(f"  weights : content={wc:.2f}  collab={wcf:.2f}  "
          f"(was {W_CONTENT:.2f}/{W_COLLAB:.2f})")
    print(f"  AP      : {apb:.4f} → {apa:.4f}  ({status})")

    lead_cols = ["track_name", "artist_name", "map_score", "hybrid_score",
                 "from_content", "from_tag", "from_neighbor"]
    print(r[lead_cols].head(5).to_string(index=False))
    print()

print("MAP re-ranking complete.")
print("Saved → data/processed/map_reranked_{user}.csv for all users.")

Running MAP re-ranking...

andy
  tier    : fallback-tier2
  weights : content=0.85  collab=0.15  (was 0.80/0.20)
  AP      : 0.0208 → 0.0218  (improved)
              track_name artist_name  map_score  hybrid_score  from_content  from_tag  from_neighbor
Running Through The Rain     YUGYEOM   0.821165      0.784124          True      True          False
               SUMMERIDE    Jay Park   0.804823      0.770534          True     False          False
            Summer Night        GRAY   0.791446      0.747298         False      True          False
             I Fancy You       Crush   0.789528      0.743239         False      True          False
All About U (Feat. Loco)     YUGYEOM   0.788899      0.749187          True      True          False

dishita
  tier    : primary
  weights : content=0.55  collab=0.45  (was 0.80/0.20)
  AP      : 0.0062 → 0.0131  (improved)
                       track_name       artist_name  map_score  hybrid_score  from_content  from_tag  from_neighbor


In [12]:
# QUANTITATIVE EVALUATION: INTRA-PLAYLIST DIVERSITY
# Measures how sonically varied each user's final re-ranked playlist is.
#
# Three complementary diversity metrics are computed per playlist:
#   1. mean_pairwise_distance : average cosine distance between all track pairs
#                               higher = more varied playlist overall
#   2. feature_std            : mean standard deviation across audio feature dimensions
#                               higher = wider spread across individual features
#   3. artist_diversity       : fraction of unique artists out of total tracks
#                               higher = less artist repetition

# TUNEABLE PARAMETERS FOR DIVERSITY SCORING
DIVERSITY_SAMPLE_CAP = 150   # max tracks to include (uses full reranked pool by default)
                             # lower this if pairwise computation becomes slow on large pools

"""
Computes mean pairwise cosine distance for a set of track vectors.
Cosine distance = 1 - cosine similarity, so 0 = identical, 1 = maximally different.
- track_vectors : (n_tracks, n_features) numpy array, already normalized
Returns a scalar mean distance across all unique pairs.
"""
def _mean_pairwise_distance(track_vectors: np.ndarray) -> float:
    # cosine_similarity returns an (n x n) matrix; we want the upper triangle only
    # to avoid counting each pair twice and exclude self-similarity on the diagonal
    sim_matrix  = cosine_similarity(track_vectors)
    n           = sim_matrix.shape[0]
    upper_idx   = np.triu_indices(n, k=1)   # k=1 skips the diagonal
    pairwise_distances = 1.0 - sim_matrix[upper_idx]
    return float(pairwise_distances.mean()) if len(pairwise_distances) > 0 else 0.0

"""
Computes mean standard deviation across all audio feature dimensions for the playlist.
Each feature (danceability, energy, valence, etc.) gets its own std; we average them.
- track_vectors : (n_tracks, n_features) numpy array
Returns a scalar representing average feature spread.
"""
def _feature_std(track_vectors: np.ndarray) -> float:
    return float(track_vectors.std(axis=0).mean())

"""
Computes artist diversity as the fraction of unique artists in the playlist.
1.0 = every track is by a different artist
0.0 = every track is by the same artist
- playlist_df : the reranked DataFrame for one user (must have artist_name column)
Returns a scalar in [0, 1].
"""
def _artist_diversity(playlist_df: pd.DataFrame) -> float:
    n_tracks  = len(playlist_df)
    n_artists = playlist_df["artist_name"].nunique()
    return float(n_artists / n_tracks) if n_tracks > 0 else 0.0

"""
Computes all three diversity metrics for a single user's re-ranked playlist.
Pulls track vectors directly from audio_matrix_norm using track_ids already
present in reranked_pools — no re-scoring or re-ranking is performed.
- user        : user name string
- playlist_df : reranked_pools[user] DataFrame from the MAP re-ranking cell
Returns a dict of diversity metrics for this user.
"""
def score_diversity(user: str, playlist_df: pd.DataFrame) -> dict:
    df = playlist_df.head(DIVERSITY_SAMPLE_CAP).copy()

    # Pull the already-normalized audio vectors for this playlist's track_ids
    track_ids    = df["track_id"].astype(int).values
    track_vectors = audio_matrix_norm[track_ids]   # shape: (n_tracks, n_audio_features)

    mpd = _mean_pairwise_distance(track_vectors)
    fsd = _feature_std(track_vectors)
    ard = _artist_diversity(df)

    return {
        "user":                   user,
        "n_tracks":               len(df),
        "mean_pairwise_distance": round(mpd, 4),
        "feature_std":            round(fsd, 4),
        "artist_diversity":       round(ard, 4),
    }

# RUN DIVERSITY SCORING FOR ALL USERS
print("Computing intra-playlist diversity...\n")
diversity_results = []

for user in USERS:
    result = score_diversity(user, reranked_pools[user])
    diversity_results.append(result)

diversity_df = pd.DataFrame(diversity_results).set_index("user")
diversity_df.to_csv(DATA_PROC / "diversity_scores.csv")

# PRINT RESULTS
print(diversity_df.to_string())
print()

# SUMMARY STATS ACROSS ALL USERS
print("Group averages:")
print(f"  mean_pairwise_distance : {diversity_df['mean_pairwise_distance'].mean():.4f}  "
      f"(std={diversity_df['mean_pairwise_distance'].std():.4f})")
print(f"  feature_std            : {diversity_df['feature_std'].mean():.4f}  "
      f"(std={diversity_df['feature_std'].std():.4f})")
print(f"  artist_diversity       : {diversity_df['artist_diversity'].mean():.4f}  "
      f"(std={diversity_df['artist_diversity'].std():.4f})")
print()
print("Most diverse playlist  :", diversity_df["mean_pairwise_distance"].idxmax())
print("Least diverse playlist :", diversity_df["mean_pairwise_distance"].idxmin())
print()
print("Diversity scoring complete.")
print("Saved → data/processed/diversity_scores.csv")

Computing intra-playlist diversity...

          n_tracks  mean_pairwise_distance  feature_std  artist_diversity
user                                                                     
andy           150                  0.0910       0.1413            0.6000
dishita        150                  0.1078       0.1533            0.6600
riya           150                  0.1101       0.1665            0.6733
priyanka       150                  0.0916       0.1488            0.6533

Group averages:
  mean_pairwise_distance : 0.1001  (std=0.0102)
  feature_std            : 0.1525  (std=0.0106)
  artist_diversity       : 0.6466  (std=0.0322)

Most diverse playlist  : riya
Least diverse playlist : andy

Diversity scoring complete.
Saved → data/processed/diversity_scores.csv


In [13]:
# QUALITATIVE EVALUATION: EXPLICIT USER FEEDBACK
# Each user selects their name, rates their top 20 recommended songs with thumbs
# up or thumbs down, and submits. Ratings are saved to data/processed/feedback_{user}.json.
# Run this cell once per user. The final re-ranking cell runs after all four have submitted.

# TUNEABLE PARAMETERS FOR FEEDBACK
FEEDBACK_N_SONGS  = 20         # number of top songs each user will rate
FEEDBACK_DIR      = DATA_PROC  # where feedback JSONs are saved (data/processed/)

# COLOUR CONSTANTS FOR THE WIDGET UI
_CLR_LIKE    = "#1DB954"   # spotify green for thumbs up
_CLR_DISLIKE = "#E74C3C"   # red for thumbs down
_CLR_NEUTRAL = "#888888"   # grey for unrated

"""
Builds the list of (track_id, track_name, artist_name) tuples for the top N songs
from a given user's re-ranked pool. These are the songs the user will rate.
"""
def _get_songs_to_rate(user: str) -> list:
    df = reranked_pools[user].head(FEEDBACK_N_SONGS)
    return [
        (int(row["track_id"]), row["track_name"], row["artist_name"])
        for _, row in df.iterrows()
    ]

"""
Saves the user's submitted ratings to data/processed/feedback_{user}.json.
Each entry records the track_id, track_name, artist_name, and rating (1=like, -1=dislike, 0=unrated).
"""
def _save_feedback(user: str, ratings: dict, songs: list):
    records = []
    for tid, track_name, artist_name in songs:
        records.append({
            "track_id":    tid,
            "track_name":  track_name,
            "artist_name": artist_name,
            "rating":      ratings.get(tid, 0),   # 1=like, -1=dislike, 0=unrated
        })
    path = FEEDBACK_DIR / f"feedback_{user}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2)
    return path

# BUILD THE WIDGET UI
# User selector dropdown
user_selector = widgets.Dropdown(
    options   = USERS,
    value     = USERS[0],
    description = "Who are you:",
    style     = {"description_width": "initial"},
    layout    = widgets.Layout(width="250px", margin="10px 0px 20px 0px"),
)

# Placeholder where song rows will be rendered
songs_output = widgets.Output()
submit_output = widgets.Output()

# Stores current ratings state: {track_id: 1 or -1}
_ratings: dict = {}
_current_songs: list = []

"""
Renders one row per song with the track name, artist, and two toggle buttons (👍 / 👎).
Buttons are mutually exclusive — selecting one deselects the other.
Called whenever the user selector changes.
"""
def _render_songs(user: str):
    global _ratings, _current_songs
    _ratings = {}
    _current_songs = _get_songs_to_rate(user)

    rows = []
    for tid, track_name, artist_name in _current_songs:
        # Truncate long names so the layout stays clean
        display_name = f"{track_name[:40]}{'…' if len(track_name) > 40 else ''}"
        display_artist = f"{artist_name[:25]}{'…' if len(artist_name) > 25 else ''}"

        label = widgets.HTML(
            value  = f"<b>{display_name}</b> &nbsp;<span style='color:#aaa'>— {display_artist}</span>",
            layout = widgets.Layout(width="420px", margin="0px 10px 0px 0px"),
        )

        like_btn    = widgets.Button(description="👍", layout=widgets.Layout(width="50px", height="32px"))
        dislike_btn = widgets.Button(description="👎", layout=widgets.Layout(width="50px", height="32px"))

        # Closure captures tid, like_btn, dislike_btn for this row
        def _make_handler(track_id, lb, db):
            def _on_like(_):
                if _ratings.get(track_id) == 1:
                    # toggle off
                    _ratings[track_id] = 0
                    lb.style.button_color = _CLR_NEUTRAL
                else:
                    _ratings[track_id] = 1
                    lb.style.button_color = _CLR_LIKE
                    db.style.button_color = _CLR_NEUTRAL
            def _on_dislike(_):
                if _ratings.get(track_id) == -1:
                    # toggle off
                    _ratings[track_id] = 0
                    db.style.button_color = _CLR_NEUTRAL
                else:
                    _ratings[track_id] = -1
                    db.style.button_color = _CLR_DISLIKE
                    lb.style.button_color = _CLR_NEUTRAL
            return _on_like, _on_dislike

        on_like, on_dislike = _make_handler(tid, like_btn, dislike_btn)
        like_btn.on_click(on_like)
        dislike_btn.on_click(on_dislike)

        row = widgets.HBox(
            [label, like_btn, dislike_btn],
            layout=widgets.Layout(margin="4px 0px"),
        )
        rows.append(row)

    with songs_output:
        clear_output(wait=True)
        display(widgets.VBox(rows))

# Submit button
submit_btn = widgets.Button(
    description  = "Submit Ratings",
    button_style = "success",
    layout       = widgets.Layout(width="160px", height="36px", margin="16px 0px 0px 0px"),
)

def _on_submit(_):
    user = user_selector.value
    path = _save_feedback(user, _ratings, _current_songs)
    n_liked    = sum(1 for v in _ratings.values() if v ==  1)
    n_disliked = sum(1 for v in _ratings.values() if v == -1)
    n_unrated  = FEEDBACK_N_SONGS - n_liked - n_disliked
    with submit_output:
        clear_output(wait=True)
        display(widgets.HTML(
            f"<div style='margin-top:10px; padding:10px; background:#1e1e1e; border-radius:6px;'>"
            f"<b style='color:{_CLR_LIKE}'>✔ Saved!</b> &nbsp;"
            f"👍 {n_liked} &nbsp; 👎 {n_disliked} &nbsp; unrated {n_unrated}<br>"
            f"<span style='color:#aaa; font-size:0.85em'>{path}</span>"
            f"</div>"
        ))

submit_btn.on_click(_on_submit)

# Re-render songs whenever a different user is selected
def _on_user_change(change):
    if change["name"] == "value":
        with submit_output:
            clear_output()
        _render_songs(change["new"])

user_selector.observe(_on_user_change, names="value")

# Initial render for the first user
_render_songs(USERS[0])

# DISPLAY THE FULL WIDGET
display(widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:4px'>🎵 Playlist Feedback</h3>"
                 "<p style='color:#aaa; margin-top:0'>Select your name, rate your songs, then hit Submit.</p>"),
    user_selector,
    songs_output,
    submit_btn,
    submit_output,
]))

In [13]:
# FINAL RE-RANKING: EXPLICIT FEEDBACK
# Loads each user's thumbs up / thumbs down ratings from feedback_{user}.json and
# produces one final re-ranked playlist per user.
#
# HOW IT WORKS:
#   - Liked tracks  → compute their audio centroid → pull ranking toward that centroid
#   - Disliked tracks → compute their audio centroid → push ranking away from that centroid
#   - Final score = map_score + (FEEDBACK_LIKE_W * like_sim) - (FEEDBACK_DISLIKE_W * dislike_sim)
#   - Tracks with a dislike rating are excluded from the final playlist entirely
#
# Requires: reranked_pools, audio_matrix_norm, track_table (all already in memory)
#           feedback_{user}.json files saved by the widget cell

# TUNEABLE PARAMETERS FOR FEEDBACK RE-RANKING
FEEDBACK_LIKE_W    = 0.30   # how much liked-track similarity boosts the final score
FEEDBACK_DISLIKE_W = 0.20   # how much disliked-track similarity penalises the final score
FEEDBACK_FINAL_K   = 100    # final playlist length after feedback re-ranking

"""
Loads a user's saved feedback JSON and returns three structures:
  - liked_ids    : set of track_ids the user thumbed up
  - disliked_ids : set of track_ids the user thumbed down
  - records      : full list of dicts (for reference / saving)
Returns (None, None, None) if the feedback file does not exist yet.
"""
def _load_feedback(user: str) -> tuple:
    path = FEEDBACK_DIR / f"feedback_{user}.json"
    if not path.exists():
        return None, None, None
    with open(path, encoding="utf-8") as f:
        records = json.load(f)
    liked_ids    = {r["track_id"] for r in records if r["rating"] ==  1}
    disliked_ids = {r["track_id"] for r in records if r["rating"] == -1}
    return liked_ids, disliked_ids, records

"""
Computes the mean audio feature centroid for a set of track_ids.
Uses audio_matrix_norm so it is on the same scale as all other similarity computations.
Returns a (1, n_features) array, or None if the id set is empty.
"""
def _centroid(track_ids: set) -> np.ndarray:
    if not track_ids:
        return None
    ids = list(track_ids)
    return audio_matrix_norm[ids].mean(axis=0, keepdims=True)   # shape: (1, n_features)

"""
Applies explicit feedback to re-score and re-rank a user's MAP-reranked pool.
Steps:
  1. Load liked / disliked track ids from feedback JSON.
  2. Compute liked centroid and disliked centroid from audio_matrix_norm.
  3. For every track in the pool, compute cosine similarity to each centroid.
  4. final_score = map_score + FEEDBACK_LIKE_W * like_sim - FEEDBACK_DISLIKE_W * dislike_sim
  5. Remove explicitly disliked tracks from the pool entirely.
  6. Sort by final_score descending, return top FEEDBACK_FINAL_K tracks.
Returns a DataFrame with all previous columns plus:
  like_sim, dislike_sim, feedback_score, feedback_applied (bool).
"""
def feedback_rerank(user: str, reranked_df: pd.DataFrame) -> pd.DataFrame:
    liked_ids, disliked_ids, records = _load_feedback(user)

    if liked_ids is None:
        print(f"  [!] No feedback file found for {user} — skipping.")
        return None

    # Compute centroids
    like_centroid    = _centroid(liked_ids)
    dislike_centroid = _centroid(disliked_ids)

    all_track_ids = reranked_df["track_id"].astype(int).values

    # Similarity of every track in the pool to the liked centroid
    if like_centroid is not None:
        like_sims = cosine_similarity(like_centroid, audio_matrix_norm[all_track_ids])[0]
    else:
        like_sims = np.zeros(len(all_track_ids))

    # Similarity of every track in the pool to the disliked centroid
    if dislike_centroid is not None:
        dislike_sims = cosine_similarity(dislike_centroid, audio_matrix_norm[all_track_ids])[0]
    else:
        dislike_sims = np.zeros(len(all_track_ids))

    result = reranked_df.copy()
    result["like_sim"]       = like_sims
    result["dislike_sim"]    = dislike_sims
    result["feedback_score"] = (
        result["map_score"]
        + FEEDBACK_LIKE_W    * result["like_sim"]
        - FEEDBACK_DISLIKE_W * result["dislike_sim"]
    )
    result["feedback_applied"] = True

    # Remove explicitly disliked tracks entirely
    result = result[~result["track_id"].astype(int).isin(disliked_ids)]

    result = (
        result
        .sort_values("feedback_score", ascending=False)
        .head(FEEDBACK_FINAL_K)
        .reset_index(drop=True)
    )

    return result

# CHECK ALL FEEDBACK FILES EXIST BEFORE RUNNING
print("Checking feedback files...\n")
missing = [u for u in USERS if not (FEEDBACK_DIR / f"feedback_{u}.json").exists()]
if missing:
    print(f"  [!] Missing feedback from: {missing}")
    print(      "      Ask them to run the feedback widget cell and submit before continuing.")
else:
    print("  All feedback files found. Running final re-ranking...\n")

    final_playlists = {}

    for user in USERS:
        final_playlists[user] = feedback_rerank(user, reranked_pools[user])
        if final_playlists[user] is None:
            continue

        final_playlists[user].to_csv(RESULTS_DIR / f"final_playlist_{user}.csv", index=False)

        liked_ids, disliked_ids, _ = _load_feedback(user)
        print(f"{user}  👍 {len(liked_ids)}  👎 {len(disliked_ids)}  "
              f"→ final playlist: {len(final_playlists[user])} tracks")

        lead_cols = ["track_name", "artist_name", "feedback_score", "map_score",
                     "like_sim", "dislike_sim"]
        print(final_playlists[user][lead_cols].head(5).to_string(index=False))
        print()

    print("Final re-ranking complete.")
    print("Saved → results/playlists/final_playlist_{user}.csv for all users.")

Checking feedback files...

  [!] Missing feedback from: ['andy', 'dishita', 'priyanka']
      Ask them to run the feedback widget cell and submit before continuing.


In [12]:
# UI-WIDGETS (?)

# TESTING

# DEMO (?)